# Tweedie causal revenue analysis

Use your actual exposure dataframe. This notebook explains preprocessing, fits the standalone causal model through the full recommendation workflow, and interprets revenue in euros and percentages. There is no synthetic application dataset. Tweedie models nonnegative means with a log link; it is not guaranteed to resist outliers or beat your existing model. All new sample snippets are collected here.

The modeled mean is `mu = exp(raw_score)`, with variance family `Var(Y | features, action) = dispersion * mu ** p`. For this workflow, `1 < p < 2`. The learner fits the conditional mean through Tweedie loss; it does not estimate a complete revenue distribution or its dispersion. The final DR stage estimates signed causal mean differences.

In [ ]:
# Find the checkout whether Jupyter started at the root or inside notebooks/.
from pathlib import Path
import sys
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'causal_dml.py').is_file():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError('Open this notebook inside the DML_CausalCustom checkout.')
sys.path.insert(0, str(REPO_ROOT))
print('Repository:', REPO_ROOT)

In [ ]:
# Install into the active notebook kernel, not another Python environment.
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT/'requirements-notebooks.txt')])

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import replace
from causal_dml import TweedieMeanRegressor, LightweightDRLearner, FinalStageConfig
from recommendation_pipeline import RecommendationConfig, run_recommendations
from pipeline import prepare_data
from IPython.display import display, Image

## 1 Load and map your real data

Each row represents the decision/exposure whose causal impact is being measured. Revenue must already be correctly attributed and mature. Keeping a user together in a split does not repair revenue duplicated across PDP views. Raw data must be in one currency and use pretreatment features.

In [ ]:
# If df is already loaded by your BigQuery/parquet workflow, keep using it.
DATA_PATH = None  # Example: '/path/to/actual_pdp_exposures.parquet'
if 'df' not in globals():
    if DATA_PATH is None:
        raise RuntimeError('Load your actual dataframe as df, or set DATA_PATH.')
    path = Path(DATA_PATH)
    df = pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)

# EDIT the LEFT side when your source columns have different names.
COLUMN_MAP = {
    # 'actual_revenue_column': 'revenue',
    # 'actual_customer_column': 'user_id',
    # 'actual_lead_column': 'lead_time',
}
source = df.rename(columns=COLUMN_MAP)
print('Input rows and columns:', source.shape)

In [ ]:
# Optional calendar features from an actual PRETREATMENT decision timestamp.
# Do not substitute a purchase/outcome timestamp. Set units from the source schema.
SOURCE_TIME_COL = None  # e.g. 'decision_time_posix'
SOURCE_TIME_UNIT = 's'  # seconds; use 'ms' only if the source is milliseconds
if SOURCE_TIME_COL is not None:
    ts = pd.to_datetime(source[SOURCE_TIME_COL], unit=SOURCE_TIME_UNIT, utc=True, errors='raise')
    source = source.assign(day=ts.dt.day, week=ts.dt.isocalendar().week.astype('int32'), month=ts.dt.month)
# Calendar extraction is not outcome-window maturity checking or temporal purging.

## 2 Declare treatments and feature roles

The default treatment is a categorical joint lead and discount action. `data_definition` and `industry` are effect modifiers X. W holds additional pretreatment confounders. `user_id` controls splitting. Product IDs can be retained as output keys; they are not treatment arms or encoded model features. For a carousel intervention, use the alternative configuration below and reassess the pretreatment status of W.

In [ ]:
# EDIT these names to the actual available and causally justified columns.
X_COLUMNS = ['data_definition', 'industry']
W_COLUMNS = ['day', 'week', 'month', 'device', 'channel']
PRODUCT_COLUMN = None  # set to 'product_id' if available

# IMPORTANT: this is a source-definition choice, not an automatic guess.
DISCOUNT_INPUT_IS_PERCENT = True  # True: 7.7 is 7.7%; False: .077 is 7.7%
REVENUE_DIVISOR = 1.0  # 1 if source is euros; 100 if source is euro cents
CURRENCY = 'EUR'       # a label, not a currency conversion service

required = ['revenue', 'user_id', 'lead_time', 'discount_percentage'] + X_COLUMNS + W_COLUMNS
if PRODUCT_COLUMN:
    required.append(PRODUCT_COLUMN)
missing = sorted(set(required)-set(source.columns))
if missing:
    raise ValueError(f'Edit mappings/features first; missing columns: {missing}')
# Copy only required columns to avoid duplicating an entire wide dataframe.
data = source[list(dict.fromkeys(required))].copy()

In [ ]:
# Convert numeric storage types such as Decimal without changing outcome semantics.
for col in ['revenue', 'lead_time', 'discount_percentage']:
    data[col] = pd.to_numeric(data[col], errors='raise').astype('float64')
if not np.isfinite(REVENUE_DIVISOR) or REVENUE_DIVISOR <= 0:
    raise ValueError('REVENUE_DIVISOR must be positive.')
data['revenue'] = data['revenue'] / REVENUE_DIVISOR
if not np.isfinite(data['revenue']).all() or (data['revenue'] < 0).any():
    raise ValueError('Tweedie needs nonnegative finite revenue. Investigate refunds instead of clipping them.')
if data['user_id'].isna().any():
    raise ValueError('Resolve missing user IDs before grouped splitting.')

# DO NOT call log1p here. Tweedie receives original-unit Y and uses a log mean link.
# DO NOT log or standardize discount before action binning.
for col in X_COLUMNS:
    data[col] = data[col].astype('string').fillna('__missing__')

display(data[['revenue', 'lead_time', 'discount_percentage']].quantile([0,.5,.9,.99,1]))
print('Zero revenue fraction:', (data['revenue'] == 0).mean())
print('Unique user groups:', data['user_id'].nunique())

## 3 Configure and inspect preparation

`log1p_outcome=False` is required for this euro-mean view. The log link is internal to the outcome nuisance. The compound Tweedie candidate uses variance power strictly between one and two. Final causal contrasts can be negative, so the final DR regressors use ordinary squared error. Candidate final settings share the same nuisance specification for selection.

In [ ]:
cfg = RecommendationConfig(
    outcome_col='revenue', group_col='user_id',
    x_cols=X_COLUMNS, report_cols=X_COLUMNS, w_cols=W_COLUMNS,
    product_col=PRODUCT_COLUMN,
    lead_col='lead_time', discount_col='discount_percentage',
    discount_input_is_percent=DISCOUNT_INPUT_IS_PERCENT,
    max_discount=.45, max_lead=30, reference='0d|0%',
    log1p_outcome=False, outcome_nuisance='tweedie',
    tweedie_variance_power=1.5, currency=CURRENCY,
    min_action_rows=100, min_cohort_action_rows=100,
    propensity_floor=.001, min_policy_propensity=.01,
    calibrate=False, cv=3, n_jobs=2,
    nuisance_y_params={'n_estimators':120, 'learning_rate':.05, 'min_child_samples':200},
    final_candidates=[
        FinalStageConfig(n_estimators=80, num_leaves=7, max_depth=3, min_child_samples=200),
        FinalStageConfig(n_estimators=120, num_leaves=15, max_depth=4, min_child_samples=300),
    ],
    scenarios=[{'name':'all', 'filters':{}},
               {'name':'lab_water', 'filters':{'data_definition':['lab_water'], 'industry':[]}}],
)

# Prepared data contains original-unit _outcome and categorical _action.
# No encoder is fitted here. Fitted encoders are learned later on training users only.
prepared, action_grid, audit = prepare_data(data,cfg)
display(prepared[['revenue','_outcome','_action']].head())
print(audit)
print('Nonzero discounts are binned AFTER percent-to-fraction conversion, when requested.')
# The complete split-specific counts are exported as preflight_actions.csv after fitting.

In [ ]:
# Alternative intervention: show a carousel versus do not show it.
# This snippet is commented so it cannot accidentally replace the joint-action analysis.
# Include the actual carousel_seen column when building data above, then use:
# cfg = replace(cfg, treatment_col='carousel_seen', categories=[0,1], reference=0)
# cfg.known_propensities = [.5,.5]  # ONLY for an actual constant 50/50 randomization
# For observational assignment, leave known_propensities=None.
# Reassess which discounts/leads/channels are pretreatment confounders for this intervention.

## 4 Fit the complete model

The pipeline reserves 50/20/10/20 of user groups for training, final-model selection, calibration, and test. It refits the selected setting on training plus selection users. Calibration is off by default; its set remains reserved. The test outcomes never select the policy. For future-time deployment, supply an externally purged temporal design; random user separation alone is not a time test.

In [ ]:
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'tweedie_revenue'
fitted, report = run_recommendations(data,cfg,OUTPUT_DIR)
print('Outcome nuisance:', report['outcome_nuisance'])
print('Link:', report['outcome_link'])
print('Tweedie power:', report['tweedie_variance_power'])
print('Original currency:', report['currency'])
display(pd.read_csv(OUTPUT_DIR/'preflight_actions.csv'))
display(pd.read_json(OUTPUT_DIR/'validation_selection.json'))

## 5 Interpret independent policy value

`policy_revenue_summary.csv` reports DR-estimated mean values for the reference and the learned fixed policy on independent test users. With no action costs, these are revenue per opportunity in euros. With costs configured, they are revenue less those costs. Percentage lift is 100 times the difference in mean values divided by the baseline mean; it is not an average of row percentages. The euro interval is an approximate clustered interval for the policy difference, not a percentage or individual-product interval.

In [ ]:
summary = pd.read_csv(OUTPUT_DIR/'policy_revenue_summary.csv')
display(summary)
e = report['policy_evaluation']
print(f"Reference mean value: {e['baseline_mean_value']:.4f} {cfg.currency} per opportunity")
print(f"Policy mean value:    {e['policy_mean_value']:.4f} {cfg.currency} per opportunity")
print(f"Incremental value:    {e['mean_gain']:+.4f} {cfg.currency} per opportunity")
if e['relative_gain_pct'] is not None:
    print(f"Relative change:      {e['relative_gain_pct']:+.2f}% of baseline mean")
else:
    print('Percentage undefined: estimated baseline is not positive.')
print('Approximate gain interval:', e['ci_low'], e['ci_high'], cfg.currency)
print('Overlap passed:', e['overlap_pass'])
print('Unsupported and clipped fractions:', e['unsupported_fraction'], e['clipped_fraction'])
print('Policy and reference ESS:', e['policy_ess'], e['baseline_ess'])
# DR mean estimates are noisy and can be negative even for nonnegative outcomes.
# Investigate weak/near-zero baselines; do not clip them merely to obtain a percentage.

A purely arithmetic illustration: a baseline mean of EUR 2.00 and action mean of EUR 2.12 correspond to EUR 0.12 lift and 6% lift. This is not a fitted result. Extrapolating to additional opportunities assumes a comparable population, an implementable policy and valid attribution; repeated PDP views cannot automatically be multiplied into independent sales.

In [ ]:
# This computes the interpretation from your ACTUAL fitted report, not example data.
if e['baseline_mean_value'] > 0:
    calculated_pct = 100*(e['policy_mean_value']/e['baseline_mean_value']-1)
    print('Ratio-of-means check:', calculated_pct)
    assert np.isclose(calculated_pct,e['relative_gain_pct'])

# A deployment-volume calculation is optional; leave unset until the unit of opportunity is valid.
ADDITIONAL_COMPARABLE_OPPORTUNITIES = None
if ADDITIONAL_COMPARABLE_OPPORTUNITIES is not None:
    print('Projected incremental value:', e['mean_gain']*ADDITIONAL_COMPARABLE_OPPORTUNITIES, cfg.currency)

## 6 Read cohort and scenario effects

The cohort table contains smoothed CATE model contrasts in original units. Its mean_effect column is euros per opportunity when Y is euro revenue. It is different from the independent policy-value estimate. Only evaluate supported actions. A positive predicted effect is not a significance test or a guaranteed increase for a particular SKU.

In [ ]:
display(pd.read_csv(OUTPUT_DIR/'cohort_effects.csv').head(30))
display(pd.read_csv(OUTPUT_DIR/'scenario_effects.csv'))
display(pd.read_csv(OUTPUT_DIR/'recommendations_test_population.csv').head(20))
if (OUTPUT_DIR/'cohort_heatmap.png').exists():
    display(Image(filename=str(OUTPUT_DIR/'cohort_heatmap.png')))

## 7 Reuse the standalone model

The pipeline bundle stores the fitted encoders, causal estimator and nuisance model. The following inspection shows the log link without changing Y. Do not exponentiate final causal effects: they are already signed euro differences. Raw outcome-nuisance predictions are conditional means, not by themselves the complete DR causal estimate.

In [ ]:
# Inspect only a small batch to limit memory. This is an API illustration, not validation.
from scipy import sparse
small = data.iloc[:20]
x_small,w_small = fitted.encode(small)
print('CATE output shape:', fitted.estimator.const_marginal_effect(x_small).shape)
print('Outcome nuisance class:', type(fitted.estimator.model_y_).__name__)

# Standalone construction for another project. Supply already encoded arrays and
# independently split users; the variables below refer to YOUR prepared arrays.
# outcome_model = TweedieMeanRegressor(variance_power=1.5,
#     model_params={'n_estimators':120,'learning_rate':.05,'num_leaves':15,
#                   'min_child_samples':200,'n_jobs':2})
# causal_model = LightweightDRLearner(categories=['reference','action'],
#     model_y=outcome_model, final_config=FinalStageConfig(), cv=3)
# causal_model.fit(Y_train_original_euros,T_train,X=X_train,W=W_train,groups=user_train)
# effect_euros = causal_model.effect(X_new,T0='reference',T1='action')
# diagnostic = causal_model.evaluate_policy(Y_test_original_euros,T_test,
#     X=X_test,W=W_test,policy=fixed_test_actions,groups=user_test)
# The final DR model is squared-error on signed pseudo-outcomes, not Tweedie.

## 8 Compare with log1p and tune cautiously

For the earlier log measurement target, explicitly select the regression nuisance and log1p_outcome=True. Its effects remain log effects, not euro differences. Keep it as a separate run/output directory. For Tweedie variance-power tuning, use training-only inner splits and compare original-unit mean predictions with the SAME scoring rule across candidates, such as Tweedie deviance at one fixed reference power. Do not compare differently constructed DR pseudo-outcome MSEs as though nuisance noise were identical, and do not select on final test revenue. Rare large orders remain a genuine source of mean-revenue uncertainty.

In [ ]:
# Separate optional log-scale run, never silently substituted for the euro result.
# log_cfg = replace(cfg,outcome_nuisance='regression',log1p_outcome=True,action_costs={})
# log_fitted,log_report = run_recommendations(data,log_cfg,REPO_ROOT/'outputs'/'log_measurement')

# Currency conversion must happen on original revenue before either run.
# Example only: if your source is cents, set REVENUE_DIVISOR=100 near the top.
# A 'EUR' label alone does not convert a mixed-currency dataset.

## References

[LightGBM objective and Tweedie variance-power parameters](https://lightgbm.readthedocs.io/en/stable/Parameters.html). See the repository REVENUE_REPORTING.md, METHODS_V2.md and RESEARCH_REVIEW.md for interpretation, DR scores, assumptions and references. The log-link nuisance targets a mean; this notebook does not fit a Tweedie dispersion parameter or produce full predictive revenue distributions.